In [ ]:
from google.colab import auth
auth.authenticate_user()

import pandas as pd
from datetime import datetime
import pytz
import time
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import zipfile
import os
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
from google.api_core.exceptions import NotFound, GoogleAPICallError
import ipywidgets as widgets
from IPython.display import display

In [2]:
# @title
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
BUCKET_NAME = "rs-nprd-dlk-ue4-gcs-ryl-sftp_generics"
RUTA= "AUTOMATIZACION CONCILIACIONES"
DATASET_ID = "produccion"
TABLE_CONCILIACION_ID= "BASES_CONCILIACION"
#TABLE_CONCILIACION_ID= "BASES_CONCILIACION_test"
TABLE_TICKET_ID= "TICKET_INFORMACION"
#TABLE_TICKET_ID= "TICKET_INFORMACION_test"
TABLE_LA_ID= "REPORTE_LA"
TABLE_HISTORICO_BASES= "HISTORICO_BASES"
TABLE_CONTROL_ID= "CONTROL_Bases_Ticket_LA"
#TABLE_CONTROL_ID= "CONTROL_Bases_Ticket_LA_test"

In [5]:
# @title
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)
zh_peru = pytz.timezone("America/Lima")

schema_Bases = [
        bigquery.SchemaField("CERTIFICADO_BANCO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_ALTA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_BAJA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("DIFERENCIA_DIAS", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("GLOSA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMERO_OPERACION", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_OPERACION", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("ORIGEN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CUENTA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FUENTE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMERO_RECLAMO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CIERRE", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("NOMBRE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_BASE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.DATE)
]

schema_Ticket_Info = [
        bigquery.SchemaField("IDEPOL", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CERT_BANCO_EXCEL", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CERTIFICADO_BANCO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODPROD", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMPOL", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CLIENTE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMCERT", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("STSCERT", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_INICIO_VIGENCIA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_FIN_VIGENCIA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_INI_ULT_COBERTURA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_FIN_ULT_COBERTURA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("CAN_LQ_COB", bigquery.enums.SqlTypeNames.INT64),
        bigquery.SchemaField("CAN_LQ_INC", bigquery.enums.SqlTypeNames.INT64),
        bigquery.SchemaField("CAN_LQ_ANU", bigquery.enums.SqlTypeNames.INT64),
        bigquery.SchemaField("CAN_LQ_EMI", bigquery.enums.SqlTypeNames.INT64),
        bigquery.SchemaField("DOC_COB", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DOC_ESTADO_COB", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA_COB", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("FECHA_VE_COB", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("DOC_ANU_INC", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DOC_ESTADO_ANU_INC", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA_INC_ANU", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("FECHA_VE_INCANU", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("PRIMA_BRUTA", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("TIPODOC", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMERO_LA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MONTO_LA", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("ESTADO_LA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CAN_LA_EMI", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMSIN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("STSSIN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_OCURR", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("MTOTOTRES", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("MTOTOTAPROB", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("MTOTOTPAGADO", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("MTOTOTPEND", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("DESCRAMO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_CLIENTE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("COD_CLIENTE_FACTURAR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CLIENTE_RESPONSABLE_PAGO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NOMBRE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.STRING)
]

schema_LA = [
        bigquery.SchemaField("PRODUCTO_RIESGO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_EMISION", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("ID_CLIENTE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_DOCUMENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NRO_LA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DOCUMENTO_SUNAT", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("COD_PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NOMBRE_PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("POLIZA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMCERT", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CERT", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MONTO", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("ESTADO_ACTUAL", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("ID_CONTRATANTE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NOMBRE_CONTRATANTE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_DOC_IDE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NRO_DOC_ID", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMOPER", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CANAL", bigquery.enums.SqlTypeNames.INT64),
        bigquery.SchemaField("TIPO_CANAL", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("GLOSA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("USUARIO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODCAJERO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMTRAMITE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("ANULADO_POR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NOMBRE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.STRING)
]

schema_historico_bases = [
        bigquery.SchemaField("FUENTE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("ORIGEN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CUENTA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_OPERACION", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("DETALLE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NRO_OPERACION", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MONTO", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("MONTO_DOLARIZADO", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("NRO_RECLAMO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CERTIFICADO_BANCO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CERTIFICADO_99", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DETALLE_DE_LOS_CASOS", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("ESTADO_DE_AVANCE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NOMBRE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING)
]

schema_Control = [
        bigquery.SchemaField("NOMBRE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.DATETIME)
]


def mover_archivo(origen, destino):
    bucket.copy_blob(origen, bucket, new_name=destino )
    origen.delete()
    return

def normalizar_dataframe(df, schema, tipo):
    columnas = [field.name for field in schema]
    for col in columnas:
        if col not in df.columns:
            if tipo == 1:
                if col in ['FECHA_ALTA','FECHA_BAJA','FECHA_OPERACION','FECHA_CIERRE']:
                    df[col]= pd.Series(pd.NaT, dtype="datetime64[ns]")
                else:
                    df[col] = None
            if tipo == 2:
                if col in ['FECHA_INICIO_VIGENCIA','FECHA_FIN_VIGENCIA','FECHA_INI_ULT_COBERTURA',
                           'FECHA_FIN_ULT_COBERTURA','FECHA_VE_COB','FECHA_VE_INCANU']:
                    df[col]= pd.Series(pd.NaT, dtype="datetime64[ns]")
                else:
                    df[col] = None
            if tipo == 3:
                if col in ['FECHA_OPERACION']:
                    df[col]= pd.Series(pd.NaT, dtype="datetime64[ns]")
                else:
                    df[col] = None
    return df[columnas]


def renombrar_col_duplicadas(columnas):
    contador = {}
    nuevas_columnas = []
    for col in columnas:
        if col in contador:
            contador[col] += 1
            nuevas_columnas.append(f"{col}_{contador[col]}")
        else:
            contador[col] = 1
            nuevas_columnas.append(col)
    return nuevas_columnas

#### FUNCION PARA COMPROBAR SI LA TABLA ESTA CREADA
def tabla_existe(table_ref):
    try:
      tabla = bigquery_client.get_table(table_ref)
      return True
    except NotFound:
        # La tabla no existe
        return False
    except GoogleAPICallError as e:
        # Otros errores de BigQuery (permisos, conexión, etc.)
        print(f"⚠️ Error al consultar BigQuery: {e}")
        raise

def borrar_registros(table_name, filename: str):
    table_ref = bigquery_client.dataset(DATASET_ID).table(table_name)
    query = f"""
            DELETE FROM `{table_ref}`
            WHERE NOMBRE_ARCHIVO = @file_name
        """
    job_config = bigquery.QueryJobConfig(query_parameters=[bigquery.ScalarQueryParameter("file_name", "STRING", filename)])
    job = bigquery_client.query(query, job_config=job_config)
    job.result()
    return

#### FUNCION PARA BUSCAR SI EL ARCHIVO YA FUE CARGADO
def buscar_archivo(filename: str) -> bool:
    # Verificar si existe la tabla de control
    table_control_ref = bigquery_client.dataset(DATASET_ID).table(TABLE_CONTROL_ID)
    if not tabla_existe(table_control_ref):
        return False
    else:
        # Buscar archivo en la tabla de control
        query = f"""
            SELECT COUNT(*) AS count
            FROM `{table_control_ref}`
            WHERE NOMBRE_ARCHIVO = @file_name
        """
        job_config = bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("file_name", "STRING", filename)])

        df = bigquery_client.query(query, job_config=job_config).to_dataframe()
        return df["count"].iloc[0] > 0

#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    job_config = bigquery.LoadJobConfig()
    if tabla_existe(table_ref):
        # Abre la tabla para agregar registros
        job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND
    else:
        tabla_tramas = bigquery.Table(table_ref, schema=schema)
        tabla_tramas = bigquery_client.create_table(tabla_tramas)
        job_config.write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE
        job_config.autodetect = False
        print(f'   ℹ️ ... Se ha creado la tabla: {table_id} en el dataset: {dataset_id} ...')
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    return

def procesar_historico_bases(lista_archivos):
    for blob in lista_archivos:
      if blob.name.endswith(".xlsx"):
        nombre_archivo = os.path.basename(blob.name)
        if not buscar_archivo(nombre_archivo):
            file = blob.download_as_string()
            if nombre_archivo == '12-03-2026 BASE CONCILIACIÓN.xlsx':
                df_base= pd.read_excel(BytesIO(file), sheet_name='BASE GENERAL', dtype=str)
                # reemplazar todo lo que no sea letra/número por _
                df_base.columns = (df_base.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True))

                df_base= df_base.rename(columns={'N__DE_OPERACI_N':'NRO_OPERACION'})
                df_base= df_base[['BASE', 'FUENTE', 'ORIGEN', 'CUENTA', 'MONEDA', 'FECHA_OPERACION', 'DETALLE',
                                                'NRO_OPERACION', 'MONTO', 'MONTO_DOLARIZADO', 'NRO_RECLAMO','CERTIFICADO_BANCO',
                                                'CERTIFICADO_99','DETALLE_DE_LOS_CASOS','ESTADO_DE_AVANCE']]

                df_base = df_base.fillna('')
                df_base['FECHA_OPERACION'] = pd.to_datetime(df_base['FECHA_OPERACION'], format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
                df_base['MONTO'] = pd.to_numeric(df_base['MONTO'], errors="coerce").astype('float64')
                df_base['MONTO_DOLARIZADO'] = pd.to_numeric(df_base['MONTO_DOLARIZADO'], errors="coerce").astype('float64')
                df_base.loc[df_base['MONEDA'].str.strip().str.lower() == 's/.', 'MONEDA'] = 'PEN'
                df_base['DETALLE_DE_LOS_CASOS']= df_base['DETALLE_DE_LOS_CASOS'].str.upper()
                df_base['ESTADO_DE_AVANCE']= df_base['ESTADO_DE_AVANCE'].str.upper()
            if nombre_archivo == 'BRIMAC_FCR1_CONTROL_ENERO_2024-2025.xlsx':
                df_base= pd.read_excel(BytesIO(file), sheet_name='BASE_2024-2025', dtype=str)
                df_base.columns = (df_base.columns.str.strip().str.upper().
                       str.replace(r'[^A-Za-z0-9]', '_', regex=True))
                df_base= df_base.rename(columns={'FECHA_INGRESO':'FECHA_OPERACION', 'CERT_BANCO':'CERTIFICADO_BANCO',
                                               'DIVISA':'MONEDA', 'IMPORTE_DIVISA_ORIGINAL_DE_PRIMAS_DEVUELTAS':'MONTO',
                                               'ESTADO':'ESTADO_DE_AVANCE'})
                df_base= df_base[['FECHA_OPERACION', 'CERTIFICADO_BANCO', 'MONEDA', 'MONTO', 'ESTADO_DE_AVANCE']]
                df_base['FECHA_OPERACION'] = pd.to_datetime(df_base['FECHA_OPERACION'], format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
                df_base['MONTO'] = pd.to_numeric(df_base['MONTO'], errors="coerce").astype('float64')
                df_base.loc[df_base['MONEDA'].str.strip().str.lower() == 'sol', 'MONEDA'] = 'PEN'
                df_base.loc[df_base['MONEDA'].str.strip().str.lower() == 'dolar', 'MONEDA'] = 'USD'
            if nombre_archivo == 'CONSOLIDADO CUADRO OBLIGACIONES - 2024 2025 2026.xlsx':
                df_base= pd.read_excel(BytesIO(file), sheet_name='Hoja1', dtype=str)
                df_base.columns = (df_base.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True))
                df_base= df_base.rename(columns={'FECHA_DE_SOLICITUD_A_OPERACIONES':'FECHA_OPERACION', 'RECLAMO':'NRO_RECLAMO',
                                               'NRO_CONTRATO__CERTIFICADO_BANCO_':'CERTIFICADO_BANCO', 'MONTO_RECLAMADO_S__USD':'MONTO',
                                               'TIPO_DE_MONEDA_DE_LA_CUENTA_BANCARIA':'MONEDA'})
                df_base= df_base[['FECHA_OPERACION','NRO_RECLAMO','CERTIFICADO_BANCO','MONEDA','MONTO']]
                df_base.loc[df_base['MONEDA'].str.strip().str.lower() == 'sol', 'MONEDA'] = 'PEN'
                df_base.loc[df_base['MONEDA'].str.strip().str.lower() != 'pen', 'MONEDA'] = 'USD'
                df_base['FECHA_OPERACION'] = pd.to_datetime(df_base['FECHA_OPERACION'], format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
                df_base['MONTO']= df_base['MONTO'].replace({',': ''}, regex=True)
                df_base['MONTO'] = pd.to_numeric(df_base['MONTO'], errors="coerce").astype('float64')
            if nombre_archivo == 'BASE DAR CONSOLIDADA.xlsx':
                df_base= pd.read_excel(BytesIO(file), sheet_name='Hoja1', dtype=str)
                df_base.columns = (df_base.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True))
                df_base= df_base.rename(columns={'CURRENCY_ID':'MONEDA', 'PREMIUM_AMOUNT':'MONTO',
                                               'FECHA_FINAL_CONTRATO___CONTRACT_CANCEL_DATE_':'FECHA_OPERACION',
                                               'ESTADO_RIMAC':'ESTADO_DE_AVANCE'})
                df_base= df_base[['FECHA_OPERACION','CERTIFICADO_BANCO','MONEDA','MONTO','ESTADO_DE_AVANCE']]
                df_base['FECHA_OPERACION'] = pd.to_datetime(df_base['FECHA_OPERACION'], format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
                df_base['MONTO']= df_base['MONTO'].replace({',': ''}, regex=True)
                df_base['MONTO'] = pd.to_numeric(df_base['MONTO'], errors="coerce").astype('float64')

            df_base['NOMBRE_ARCHIVO']= nombre_archivo
            df_base = normalizar_dataframe(df_base, schema_historico_bases, 3)
            fecha_carga = datetime.now(zh_peru).date()
            df_control = pd.DataFrame([{"NOMBRE_ARCHIVO": nombre_archivo, "FECHA_CARGA": fecha_carga}])

            ##### Guardar dataframes en BigQuery
            Guardar_en_BigQuery(df_base, DATASET_ID, TABLE_HISTORICO_BASES, schema_historico_bases)
            Guardar_en_BigQuery(df_control, DATASET_ID, TABLE_CONTROL_ID, schema_Control)
            print(f"   ✅  ARCHIVO - {nombre_archivo} - CARGADO ...")
        else:
            print(f"   ⚠️  EL ARCHIVO - {nombre_archivo} - YA FUE CARGADO ANTERIORMENTE ...")
    return

def procesar_archivo_reemplazar(lista_archivos, carpeta):
    if carpeta in ['RECLAMOS','TICKET INFORMACION']:
      for blob in lista_archivos:
        if blob.name.endswith(".xlsx"):
          nombre_archivo = os.path.basename(blob.name)
          file = blob.download_as_string()
          if carpeta == 'RECLAMOS':
              if buscar_archivo(nombre_archivo):
                  print(f"   ⏳ ... ELIMINANDO REGISTROS ...")
                  borrar_registros(TABLE_CONCILIACION_ID, nombre_archivo)
                  print(f"   ✅  REGISTROS ELIMINADOS CORRECTAMENTE ...")
              print(f"   ⏳ ... CARGANDO ARCHIVO: {nombre_archivo} ...")
              df_datos= pd.read_excel(BytesIO(file), sheet_name='DEVOLUCIONES GENERALES', dtype=str)
              df_datos.columns = (df_datos.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True))
              df_datos= df_datos.rename(columns={'NRO_CONTRATO__CERTIFICADO_BANCO_':'CERTIFICADO_BANCO',
                               'TIPO_DE_MONEDA_DE_LA_CUENTA_BANCARIA':'MONEDA', 'MONTO_RECLAMADO_S__USD':'PRIMA',
                               'FECHA_DE_SOLICITUD_A_OPERACIONES':'FECHA_OPERACION',
                               'FECHA_DE_CIERRE':'FECHA_CIERRE','RECLAMO':'NUMERO_RECLAMO'})
              df_datos = df_datos.loc[:, ~df_datos.columns.duplicated()]
              df_datos['PRODUCTO'] = df_datos['PRODUCTO'].str.upper()
              df_datos['PRIMA'] = df_datos['PRIMA'].replace({',': ''}, regex=True)
              df_datos['PRIMA'] = pd.to_numeric(df_datos['PRIMA'], errors="coerce").astype('float64')
              df_datos['FECHA_OPERACION'] = pd.to_datetime(df_datos['FECHA_OPERACION'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.normalize()
              df_datos['FECHA_CIERRE'] = pd.to_datetime(df_datos['FECHA_CIERRE'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.normalize()
              df_datos.loc[df_datos['MONEDA'].str.strip().str.lower() == 'sol', 'MONEDA'] = 'PEN'
              #df_datos.loc[df_datos['MONEDA'].fillna('').str.strip().str.lower()=='s/.', 'MONEDA'] = 'PEN'
              df_datos.loc[df_datos['MONEDA'].str.strip().str.lower() == 'dolar', 'MONEDA'] = 'USD'
              df_datos= df_datos[['CERTIFICADO_BANCO','PRODUCTO','MONEDA','PRIMA','FECHA_OPERACION','NUMERO_RECLAMO','FECHA_CIERRE']]
              #df_datos= df_datos[['ID','CERTIFICADO_BANCO','PRODUCTO','MONEDA','PRIMA','FECHA_OPERACION','NUMERO_RECLAMO']]

              fecha_carga = datetime.now(zh_peru).date()
              df_datos['NOMBRE_ARCHIVO']= nombre_archivo
              df_datos['TIPO_BASE'] = 'RECLAMOS'
              df_datos['FECHA_CARGA'] = fecha_carga
              df_datos = normalizar_dataframe(df_datos, schema_Bases, 1)
              ##### Guardar dataframes en BigQuery
              Guardar_en_BigQuery(df_datos, DATASET_ID, TABLE_CONCILIACION_ID, schema_Bases)
              if not buscar_archivo(nombre_archivo):
                  df_control = pd.DataFrame([{"NOMBRE_ARCHIVO": nombre_archivo, "FECHA_CARGA": fecha_carga}])
                  Guardar_en_BigQuery(df_control, DATASET_ID, TABLE_CONTROL_ID, schema_Control)
              print(f"   ✅  ARCHIVO - {nombre_archivo} - CARGADO ...")
          if carpeta == 'TICKET INFORMACION':
              if buscar_archivo(nombre_archivo):
                  print(f"   ⏳ ... ELIMINANDO REGISTROS ...")
                  borrar_registros(TABLE_TICKET_ID, nombre_archivo)
                  print(f"   ✅  REGISTROS ELIMINADOS CORRECTAMENTE ...")
              print(f"   ⏳ ... CARGANDO ARCHIVO: {nombre_archivo} ...")
              df_datos= pd.read_excel(BytesIO(file), sheet_name=0, dtype=str)
              df_datos.columns = (df_datos.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True))
              df_datos= df_datos.rename(columns={'NRO_CERT_BANCO':'CERTIFICADO_BANCO', 'TIPO_SEGURO':'PRODUCTO',
                                  'FEC_INI_VIG_CERT':'FECHA_INICIO_VIGENCIA', 'FEC_FIN_VIG_CERT':'FECHA_FIN_VIGENCIA',
                                  'FECVE_COB':'FECHA_VE_COB', 'FECVE_COB_1':'FECHA_VE_INCANU','FECOCURR':'FECHA_OCURR',
                                  'DOCUMENTO_COB':'DOC_COB', 'DOCUMENTO_ESTADO_COB':'DOC_ESTADO_COB',
                                  'DOCUMENTO_ANU_INC':'DOC_ANU_INC','DOCUMENTO_ESTADO_ANU_INC':'DOC_ESTADO_ANU_INC',
                                  'P_BRUTA':'PRIMA_BRUTA','MTO_LA':'MONTO_LA'})
              df_datos['FECHA_INICIO_VIGENCIA'] = pd.to_datetime(df_datos['FECHA_INICIO_VIGENCIA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
              df_datos['FECHA_FIN_VIGENCIA'] = pd.to_datetime(df_datos['FECHA_FIN_VIGENCIA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
              df_datos['FECHA_INI_ULT_COBERTURA'] = pd.to_datetime(df_datos['FECHA_INI_ULT_COBERTURA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
              df_datos['FECHA_FIN_ULT_COBERTURA'] = pd.to_datetime(df_datos['FECHA_FIN_ULT_COBERTURA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
              df_datos['FECHA_VE_COB'] = pd.to_datetime(df_datos['FECHA_VE_COB'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
              df_datos['FECHA_VE_INCANU'] = pd.to_datetime(df_datos['FECHA_VE_INCANU'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
              df_datos['FECHA_OCURR'] = pd.to_datetime(df_datos['FECHA_OCURR'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
              df_datos['CAN_LQ_COB'] = pd.to_numeric(df_datos['CAN_LQ_COB'], errors='coerce').astype('Int64')
              df_datos['CAN_LQ_INC'] = pd.to_numeric(df_datos['CAN_LQ_INC'], errors='coerce').astype('Int64')
              df_datos['CAN_LQ_ANU'] = pd.to_numeric(df_datos['CAN_LQ_ANU'], errors='coerce').astype('Int64')
              df_datos['CAN_LQ_EMI'] = pd.to_numeric(df_datos['CAN_LQ_EMI'], errors='coerce').astype('Int64')
              df_datos['PRIMA_COB'] = pd.to_numeric(df_datos['PRIMA_COB'], errors="coerce").astype('float64')
              df_datos['PRIMA_INC_ANU'] = pd.to_numeric(df_datos['PRIMA_INC_ANU'], errors="coerce").astype('float64')
              df_datos['PRIMA_BRUTA'] = pd.to_numeric(df_datos['PRIMA_BRUTA'], errors="coerce").astype('float64')
              df_datos['MONTO_LA'] = pd.to_numeric(df_datos['MONTO_LA'], errors="coerce").astype('float64')
              df_datos['MTOTOTRES'] = pd.to_numeric(df_datos['MTOTOTRES'], errors="coerce").astype('float64')
              df_datos['MTOTOTAPROB'] = pd.to_numeric(df_datos['MTOTOTAPROB'], errors="coerce").astype('float64')
              df_datos['MTOTOTPAGADO'] = pd.to_numeric(df_datos['MTOTOTPAGADO'], errors="coerce").astype('float64')
              df_datos['MTOTOTPEND'] = pd.to_numeric(df_datos['MTOTOTPEND'], errors="coerce").astype('float64')

              fecha_carga = datetime.now(zh_peru).date()
              df_datos['NOMBRE_ARCHIVO']= nombre_archivo
              df_datos['TIPO'] = 'ACSELX'
              df_datos['FECHA_CARGA']= fecha_carga
              df_datos = normalizar_dataframe(df_datos, schema_Ticket_Info,2) # Normalizar la tabla
              Guardar_en_BigQuery(df_datos, DATASET_ID, TABLE_TICKET_ID, schema_Ticket_Info)
              if not buscar_archivo(nombre_archivo):
                  df_control = pd.DataFrame([{"NOMBRE_ARCHIVO": nombre_archivo, "FECHA_CARGA": fecha_carga}])
                  Guardar_en_BigQuery(df_control, DATASET_ID, TABLE_CONTROL_ID, schema_Control)
              print(f"   ✅  ARCHIVO - {nombre_archivo} - CARGADO ...")

          # Mover archivos
          destino= f"{RUTA}/{carpeta}/{nombre_archivo}"
          mover_archivo(blob, destino)
    if carpeta == 'REPORTE EMISION SATA':
        for blob in lista_archivos:
            if blob.name.endswith(".txt"):
              nombre_archivo = os.path.basename(blob.name)
              file = blob.download_as_string()
              if buscar_archivo(nombre_archivo):
                  print(f"   ⏳ ... ELIMINANDO REGISTROS ...")
                  borrar_registros(TABLE_TICKET_ID, nombre_archivo)
                  print(f"   ✅  REGISTROS ELIMINADOS CORRECTAMENTE ...")
              print(f"   ⏳ ... CARGANDO ARCHIVO: {nombre_archivo} ...")
              df_datos= pd.read_table(BytesIO(file),  dtype=str, encoding="latin1")
              df_datos.columns = (df_datos.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True))
              df_datos= df_datos.rename(columns={
                                'NOMBRE_PRODUCTO':'PRODUCTO', 'PRODUCTO_ACSELX':'CODPROD',
                                'NRO_POLIZA':'NUMPOL', 'NOMBRE_CLIENTE':'CLIENTE',
                                'ESTADO_POLIZA':'STSCERT', 'FECHA_FIN_VIGENCIA_POLIZA':'FECHA_FIN_VIGENCIA',
                                'FECHA_INICIO_COBERTURA':'FECHA_INI_ULT_COBERTURA',
                                'FECHA_FIN_COBERTURA':'FECHA_FIN_ULT_COBERTURA', 'PRIMA_BRUTA':'PRIMA_COB'})
              df_datos['FECHA_FIN_VIGENCIA'] = pd.to_datetime(df_datos['FECHA_FIN_VIGENCIA'],format='%d/%m/%Y %H:%M:%S', errors='coerce').dt.date
              df_datos['FECHA_INI_ULT_COBERTURA'] = pd.to_datetime(df_datos['FECHA_INI_ULT_COBERTURA'],format='%d/%m/%Y %H:%M:%S', errors='coerce').dt.date
              df_datos['FECHA_FIN_ULT_COBERTURA'] = pd.to_datetime(df_datos['FECHA_FIN_ULT_COBERTURA'],format='%d/%m/%Y %H:%M:%S', errors='coerce').dt.date
              df_datos['PRIMA_COB'] = pd.to_numeric(df_datos['PRIMA_COB'], errors="coerce").astype('float64')
              df_datos['NUMPOL'] = df_datos['NUMPOL'].str.lstrip('0')
              df_datos = df_datos[df_datos['TIPO_LQ_LA'].isin(['BOLETA DE VENTA', 'FACTURA'])]

              fecha_carga = datetime.now(zh_peru).date()
              df_datos['NOMBRE_ARCHIVO']= nombre_archivo
              df_datos['TIPO'] = 'SATA'
              df_datos['FECHA_CARGA']= fecha_carga
              df_datos = normalizar_dataframe(df_datos, schema_Ticket_Info,2)
              Guardar_en_BigQuery(df_datos, DATASET_ID, TABLE_TICKET_ID, schema_Ticket_Info)
              if not buscar_archivo(nombre_archivo):
                  df_control = pd.DataFrame([{"NOMBRE_ARCHIVO": nombre_archivo, "FECHA_CARGA": fecha_carga}])
                  Guardar_en_BigQuery(df_control, DATASET_ID, TABLE_CONTROL_ID, schema_Control)
              print(f"   ✅  ARCHIVO - {nombre_archivo} - CARGADO ...")
    if carpeta == 'LA':
        for blob in lista_archivos:
            if blob.name.endswith(".csv"):
              nombre_archivo = os.path.basename(blob.name)
              file = blob.download_as_string()
              if buscar_archivo(nombre_archivo):
                  print(f"   ⏳ ... ELIMINANDO REGISTROS ...")
                  borrar_registros(TABLE_LA_ID, nombre_archivo)
                  print(f"   ✅  REGISTROS ELIMINADOS CORRECTAMENTE ...")
              print(f"   ⏳ ... CARGANDO ARCHIVO: {nombre_archivo} ...")
              df_datos= pd.read_csv(BytesIO(file), dtype=str)
              df_datos.columns = (df_datos.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True))
              df_datos = df_datos.drop(columns=['ANIO','ANO','FECHA_EMI'])
              df_datos = df_datos.rename(columns={'PRODUCTO':'COD_PRODUCTO', 'DOCUMENTO':'NRO_LA', 'DESC_PRODUCTO':'NOMBRE_PRODUCTO',
                              'DOCIDENT_NUMERO':'NRO_DOC_ID','DES_TIPOCANAL':'TIPO_CANAL'})
              df_datos = df_datos.fillna('')
              #df_datos['FECHA_EMISION'] = pd.to_datetime(df_datos['FECHA_EMISION'],format='%d/%m/%Y', errors='coerce').dt.date
              df_datos['FECHA_EMISION'] = pd.to_datetime(df_datos['FECHA_EMISION'],format='%Y-%m-%d', errors='coerce').dt.date
              df_datos['MONTO'] = pd.to_numeric(df_datos['MONTO'], errors="coerce").astype('float64')
              df_datos['CANAL'] = pd.to_numeric(df_datos['CANAL'], errors="coerce").astype('Int64')
              df_datos.loc[df_datos['MONEDA'].str.strip().str.lower() == 'sol', 'MONEDA'] = 'PEN'
              df_datos.loc[df_datos['MONEDA'].str.strip().str.lower() == 'dolar', 'MONEDA'] = 'USD'
              df_datos['TIPO'] = df_datos['TIPO'].str.upper()
              df_datos['CERT']= df_datos['CERT'].str.replace('.0', '', regex=False)
              df_datos['SISTEMA'] = np.select(
                      [
                          df_datos['COD_PRODUCTO'].str[:1] == '8',   # primer caracter
                          df_datos['COD_PRODUCTO'].str[:2] == '41'   # primeros 2 caracteres
                      ],
                      ['ACSELE','RIMACSALUD'],
                      default='ACSELX'
              )
              df_datos['CONCATENADO'] = np.select(
                      [
                          df_datos['COD_PRODUCTO'].str[:2] == '41',
                          df_datos['COD_PRODUCTO'].isin(['8308','8835','8045','8046']),
                          df_datos['COD_PRODUCTO'].str.startswith('8')
                      ],
                      [
                          df_datos['COD_PRODUCTO'] + '-' + df_datos['POLIZA'],
                          df_datos['COD_PRODUCTO'] + '-' + df_datos['POLIZA'],
                          df_datos['COD_PRODUCTO'] + '-' + df_datos['NUMCERT']
                      ],
                      default=df_datos['COD_PRODUCTO'] + '-' + df_datos['POLIZA'] + '-' + df_datos['CERT']
              )
              fecha_carga = datetime.now(zh_peru).date()
              df_datos['NOMBRE_ARCHIVO']= nombre_archivo
              df_datos['FECHA_CARGA']= fecha_carga
              Guardar_en_BigQuery(df_datos, DATASET_ID, TABLE_LA_ID, schema_LA)
              if not buscar_archivo(nombre_archivo):
                  df_control = pd.DataFrame([{"NOMBRE_ARCHIVO": nombre_archivo, "FECHA_CARGA": fecha_carga}])
                  Guardar_en_BigQuery(df_control, DATASET_ID, TABLE_CONTROL_ID, schema_Control)
              if carpeta == 'RECLAMOS':
                  # Mover archivos
                  destino= f"{RUTA}/{carpeta}/{nombre_archivo}"
                  mover_archivo(blob, destino)
              print(f"   ✅  ARCHIVO - {nombre_archivo} - CARGADO ...")
    return

def procesar_archivos(lista_archivos, carpeta):
    for blob in lista_archivos:
      if blob.name.endswith(".xlsx"):
        nombre_archivo = os.path.basename(blob.name)
        if not buscar_archivo(nombre_archivo):
            print(f"   ⏳ ... CARGANDO ARCHIVO: {nombre_archivo} ...")
            file = blob.download_as_string()
            df_base= pd.read_excel(BytesIO(file), sheet_name=0, dtype=str)
            # reemplazar todo lo que no sea letra/número por _
            df_base.columns = (df_base.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True))
            if carpeta == 'DAR':
                df_base.drop(['CANAL','MES','CLIENTE','EMISI_N'], axis=1, inplace=True)
                df_base= df_base.rename(columns={'CONTRATO':'CERTIFICADO_BANCO', 'PRD':'CODIGO_PRODUCTO', 'FEC_ALTA':'FECHA_ALTA',
                                      'FEC_BAJA':'FECHA_BAJA', 'DIAS':'DIFERENCIA_DIAS','DIV':'MONEDA'})
                df_base['FECHA_ALTA']= pd.to_datetime(df_base['FECHA_ALTA'],format='%Y-%m-%d', errors='coerce').dt.date
                df_base['FECHA_BAJA']= pd.to_datetime(df_base['FECHA_BAJA'],format='%Y-%m-%d', errors='coerce').dt.date
                df_base['DIFERENCIA_DIAS'] = (pd.to_datetime(df_base['FECHA_BAJA']) - pd.to_datetime(df_base['FECHA_ALTA'])).dt.days
                df_base['PRIMA'] = pd.to_numeric(df_base['PRIMA'], errors="coerce").astype('float64')
            if carpeta == 'FCR1':
                df_base = df_base.loc[:, ~df_base.columns.duplicated()]
                df_base= df_base.rename(columns={'N_MERO_DE_CONTRATO_DE_SEGURO':'CERTIFICADO_BANCO', 'TIPO_DE_SEGURO':'PRODUCTO', 'DIVISA':'MONEDA',
                                      'IMPORTE_ORIGINAL':'PRIMA'})
                df_base= df_base[['CERTIFICADO_BANCO','PRODUCTO','MONEDA','PRIMA','GLOSA']]
                df_base.loc[df_base['MONEDA'].str.strip().str.lower() == 'sol', 'MONEDA'] = 'PEN'
                df_base.loc[df_base['MONEDA'].str.strip().str.lower() == 'dolar', 'MONEDA'] = 'USD'
                df_base['CERTIFICADO_BANCO'] = (df_base['CERTIFICADO_BANCO'].str.strip().str.replace('-', '', regex=False))
                df_base['PRIMA'] = pd.to_numeric(df_base['PRIMA'], errors="coerce").astype('float64')
            if carpeta == 'FCR2':
                df_base= df_base.rename(columns={'NRO__CONTRATO':'CERTIFICADO_BANCO', 'DIVISA':'MONEDA',
                                        'SALDO':'PRIMA', 'GLOSA_1':'GLOSA', 'NUMERO_DE_CONTROL':'NUMERO_OPERACION'})
                df_base= df_base[['CERTIFICADO_BANCO','MONEDA','PRIMA','GLOSA','NUMERO_OPERACION']]
                df_base['CERTIFICADO_BANCO'] = (df_base['CERTIFICADO_BANCO'].str.strip().str.replace('-', '', regex=False))
                df_base['PRIMA'] = pd.to_numeric(df_base['PRIMA'], errors="coerce").astype('float64')
            if carpeta == 'OMX':
                df_base.columns = renombrar_col_duplicadas(df_base.columns)
                df_base = df_base.drop(columns=['PRODUCTO'])
                df_base = df_base.rename(columns={'CERTIFICADO_BCO_ORIGINAL':'CERTIFICADO_BANCO',
                                      'MONTO':'PRIMA', 'DETALLE':'GLOSA', 'N__DE_OPERACION':'NUMERO_OPERACION',
                                      '_RECLAMO':'NUMERO_RECLAMO','PRODUCTO_2':'PRODUCTO'})
                df_base['PRODUCTO'] = df_base['PRODUCTO'].str.upper()
                df_base['PRIMA'] = pd.to_numeric(df_base['PRIMA'], errors="coerce").astype('float64')
                df_base['FECHA_OPERACION'] = pd.to_datetime(df_base['FECHA_OPERACION'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.normalize()
                df_base.loc[df_base['MONEDA'].str.strip().str.lower() == 'S/.', 'MONEDA'] = 'PEN'
                df_base.loc[df_base['MONEDA'].str.strip().str.lower() == 'dolar', 'MONEDA'] = 'USD'
                df_base= df_base[['CERTIFICADO_BANCO','PRODUCTO','MONEDA','PRIMA','GLOSA','NUMERO_OPERACION',
                        'FECHA_OPERACION','ORIGEN','CUENTA','FUENTE','NUMERO_RECLAMO']]

            if carpeta == 'PIC':
                df_base = df_base.loc[:, ~df_base.columns.duplicated()]
                df_base= df_base.rename(columns={'N_MERO_DE_CONTRATO_DE_SEGURO':'CERTIFICADO_BANCO',
                                'TIPO_DE_SEGURO':'PRODUCTO', 'DIVISA':'MONEDA',
                                'IMPORTE_ORIGINAL':'PRIMA'})
                df_base= df_base[['CERTIFICADO_BANCO','PRODUCTO','MONEDA','PRIMA','GLOSA']]
                df_base['CERTIFICADO_BANCO'] = (df_base['CERTIFICADO_BANCO'].str.strip().str.replace('-', '', regex=False))
                df_base['PRODUCTO'] = df_base['PRODUCTO'].str.upper()
                df_base['PRIMA'] = pd.to_numeric(df_base['PRIMA'], errors="coerce").astype('float64')
                df_base.loc[df_base['MONEDA'].str.strip().str.lower() == 'sol', 'MONEDA'] = 'PEN'
                df_base.loc[df_base['MONEDA'].str.strip().str.lower() == 'dolar', 'MONEDA'] = 'USD'
            if carpeta == 'PT GLOMO':
                df_base= df_base.rename(columns={'NRO_CERT_BANCO':'CERTIFICADO_BANCO','FECHA_ALTA_INICIAL':'FECHA_ALTA',
                                         'FECHA_CANCELACION_EN_BANCO':'FECHA_BAJA','PRIMA_EMITIDA':'PRIMA'})
                df_base['FECHA_ALTA'] = pd.to_datetime(df_base['FECHA_ALTA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
                df_base['FECHA_BAJA'] = pd.to_datetime(df_base['FECHA_BAJA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
                df_base['PRIMA'] = pd.to_numeric(df_base['PRIMA'], errors="coerce").astype('float64')
                df_base['PRODUCTO'] = 'PT ROYAL'
                df_base['MONEDA'] = 'PEN'
                df_base= df_base[['CERTIFICADO_BANCO','PRODUCTO','FECHA_ALTA','FECHA_BAJA','MONEDA','PRIMA']]

            fecha_carga = datetime.now(zh_peru).date()
            df_base['NOMBRE_ARCHIVO']= nombre_archivo
            df_base['TIPO_BASE'] = carpeta
            df_base['FECHA_CARGA'] = fecha_carga
            df_control = pd.DataFrame([{"NOMBRE_ARCHIVO": nombre_archivo, "FECHA_CARGA": fecha_carga}])
            df_base = normalizar_dataframe(df_base, schema_Bases,1)

            ##### Guardar dataframes en BigQuery
            Guardar_en_BigQuery(df_base, DATASET_ID, TABLE_CONCILIACION_ID, schema_Bases)
            Guardar_en_BigQuery(df_control, DATASET_ID, TABLE_CONTROL_ID, schema_Control)

            if carpeta == 'FCR2':
                # Mover archivos
                destino= f"{RUTA}/{carpeta}/{nombre_archivo}"
                mover_archivo(blob, destino)
            print(f"   ✅  ARCHIVO - {nombre_archivo} - CARGADO ...")
        else:
            print(f"   ⚠️  EL ARCHIVO - {nombre_archivo} - YA FUE CARGADO ANTERIORMENTE ...")
    return

############### PROCESO DE CARGA DE ARCHIVOS EXCEL A BIGQUERY ##########
bucket = storage_client.bucket(BUCKET_NAME)

#------------------------------ WIDGETS VISUALES ------------------------------------------
lista1 = ['DAR', 'FCR1', 'FCR2','OMX','PIC','PT GLOMO']
lista2 = ['RECLAMOS', 'TICKET INFORMACION', 'REPORTE EMISION SATA','LA']
#label = widgets.Label("Seleccione opciones:")
checkboxes1 = [
    widgets.Checkbox(value=False, description=op)
    for op in lista1]

checkboxes2 = [
    widgets.Checkbox(value=False, description=op)
    for op in lista2]

contenedor = widgets.GridBox(
    checkboxes1 + checkboxes2,
    layout=widgets.Layout(grid_template_columns="repeat(5, 200px)")
)
boton = widgets.Button(
    description="PROCESAR",
    button_style='success'
)
#----------------------------------------------------------------------------------------
def ejecutar(b):
    #fi = fecha_inicio.value
    #ff = fecha_fin.value
    lista_bases1 = [cb.description for cb in checkboxes1 if cb.value]
    lista_bases2 = [cb.description for cb in checkboxes2 if cb.value]

    # Validaciones
    #if fi is None or ff is None:
    #    print("❌ Debes seleccionar ambas fechas")
    #    return

    if not lista_bases1 and not lista_bases2:
        print("❌ Debes seleccionar al menos 1 Base")
        return

    for folder in lista_bases1:
        if folder == 'FCR2':
          ruta= f"{RUTA}/{folder}/NUEVOS/"
        else:
          ruta= f"{RUTA}/{folder}/"
        blobs_excels = list(bucket.list_blobs(prefix= ruta))
        print(f"ℹ️ CARPETA {folder}:")
        procesar_archivos(blobs_excels, folder)

    for folder in lista_bases2:
        if folder in ['RECLAMOS','TICKET INFORMACION']:
          ruta= f"{RUTA}/{folder}/NUEVOS/"
        else:
          ruta= f"{RUTA}/{folder}/"
        blobs_excels = list(bucket.list_blobs(prefix= ruta))
        print(f"ℹ️ CARPETA {folder}:")
        procesar_archivo_reemplazar(blobs_excels, folder)
    print(f"ℹ️ ---------- CARGA COMPLETADA ---------- ℹ️")
    return

boton.on_click(ejecutar)

panel = widgets.VBox([
    widgets.HTML("<h3>Seleccione las Bases:</h3>"),
    #fecha_inicio,
    #fecha_fin,
    contenedor,
    boton
])

display(panel)


ℹ️ CARPETA RECLAMOS:
   ⏳ ... ELIMINANDO REGISTROS ...
   ✅  REGISTROS ELIMINADOS CORRECTAMENTE ...
   ⏳ ... CARGANDO ARCHIVO: 2026 - Abril a Junio.xlsx ...
   ✅  ARCHIVO - 2026 - Abril a Junio.xlsx - CARGADO ...
ℹ️ ---------- CARGA COMPLETADA ---------- ℹ️


#PRUEBAS

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df_datos= pd.read_csv("/content/LA_2026.csv", dtype=str)
df_datos.columns = (df_datos.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True))

In [ ]:
df_datos['FECHA_EMISION'] = pd.to_datetime(df_datos['FECHA_EMISION'],format='%Y-%m-%d', errors='coerce').dt.date
df_datos['MONTO'] = pd.to_numeric(df_datos['MONTO'], errors="coerce").astype('float64')

In [ ]:
df_datos.head(3)

,ANIO,PRODUCTO_RIESGO,TIPO,FECHA_EMISION,ANO,ID_CLIENTE,TIPO_DOCUMENTO,DOCUMENTO,DOCUMENTO_SUNAT,PRODUCTO,...,DOCIDENT_NUMERO,NUMOPER,CANAL,DES_TIPOCANAL,GLOSA,USUARIO,CODCAJERO,NUMTRAMITE,ANULADO_POR,FECHA_EMI
0,2026,BBVA,Liquidacion de Abono,2026-06-05,2026,3822636,LA,179345229,NaN,2101,...,10409831511,4771458171,NaN,NaN,NaN,PAREJA CHUQUIZANA ANGELA MAGNOLIA,MVALDEZ,NaN,NaN,2026-06-05
1,2026,BBVA,Liquidacion de Abono,2026-06-05,2026,4131277,LA,179345889,NaN,2101,...,43281912,4771481962,NaN,NaN,NaN,FERREYRA SALVATIERRA LUIS ALBERTO,MVALDEZ,NaN,NaN,2026-06-05
2,2026,BBVA,Liquidacion de Abono,2026-06-06,2026,34340013,LA,685813320,NaN,8308,...,09151351,505190640139,NaN,NaN,NaN,PORTILLO OQUENDO GLORIA MARIA,MVALDEZ,NaN,NaN,2026-06-06


In [ ]:
bucket = storage_client.bucket(BUCKET_NAME)
path= f"{RUTA}/RECLAMOS/NUEVOS"
blob_list = list(bucket.list_blobs(prefix=path))
for blob in blob_list:
    if blob.name.endswith(".xlsx"):
      print(f"⏳ ... PROCESANDO ARCHIVO: {blob.name} ... ⏳")

⏳ ... PROCESANDO ARCHIVO: AUTOMATIZACION CONCILIACIONES/RECLAMOS/NUEVOS/Marzo del 2023 - Febrero del 2024.xlsx ... ⏳


In [ ]:
file = blob.download_as_string()

In [ ]:
df_datos= pd.read_excel(BytesIO(file), sheet_name='DEVOLUCIONES GENERALES', dtype=str)
df_datos.columns = (df_datos.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True))

In [ ]:
df_datos.head(3)

,N_MERO_DEL_CASO_SF,TIPO_DE_DEVOLUCION,FECHA_DE_ANULACI_N,CANTIDAD_DE_PRIMAS_A_DEVOLVER,_PRORRATA_,POLIZA__RIMAC_,FECHA_DE_SOLICITUD_A_OPERACIONES,REQUERIMIENTO,CLIENTE,ASISTENTE,...,FECHA_DE_OPERACI_N,ESTADO,OBSERVACIONES_OPERACIONES,RESPONSABLE,DNI,POLIZA,COD_CLIENTE,VIG_INI,VIG_FIN,IMPORTE2
0,19745497,UNICO,NaN,1,NO,NaN,2023-03-01 00:00:00,5042301491,ANA BEATRIZ ESPINOZA ALIAGA,PAOLA JANET BEDOYA OLIVEROS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0019635266,UNICO,NaN,1,NO,NaN,2023-03-01 00:00:00,20032300191,CARLOS ARTURO FLORES PERAZA,WILDER MENESES HUACCHARAQUI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0019512263,UNICO,NaN,1,NO,NaN,2023-03-02 00:00:00,01032301752,DIANA PATRICIA GARRIDO MATTA,WILDER MENESES HUACCHARAQUI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
